## DATA CLEANING- OFSTED

In [1]:
#import libraries
import pandas as pd

# Load dataset
df = pd.read_csv(
    '../data_raw/Management_information_-_childcare_providers_and_inspections_-_most_recent_inspections_data_as_at_31_December_2023.csv',
    skiprows=2,low_memory=False
)

In [2]:
#inspect data
df.head()

,Provider URN,Registration Date,Provider Type,Provider Subtype,Provider Status,Individual Register Combinations,Provider Early Years Register Flag,Provider Compulsory Childcare Register Flag,Provider Voluntary Childcare Register Flag,Registered Person URN,...,3rd Most Recent Full: Overall Effectiveness,NCOR: Inspection Number,NCOR: Inspection Date,NCOR: Overall Effectiveness,OOSC: Inspection Number,OOSC: Inspection Date,OOSC: Overall Effectiveness,CR: Inspection Number,CR: Inspection Date,CR: Overall Effectiveness
0,109902,01/10/1988,Childcare on non-domestic premises,Full day care,Active,ALL,Y,Y,Y,RP911587,...,2.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,EY471265,16/12/2013,Childcare on non-domestic premises,Full day care,Active,ALL,Y,Y,Y,RP910789,...,2.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2594358,17/07/2020,Childcare on non-domestic premises,Full day care,Active,EYR only,Y,N,N,2594357,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,EY367483,23/01/2008,Childcare on non-domestic premises,Full day care,Active,ALL,Y,Y,Y,RP527613,...,2.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,201573,23/11/1976,Childminder,NaN,Active,ALL,Y,Y,Y,NaN,...,4.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [3]:
#view columns
df.columns

Index(['Provider URN', 'Registration Date', 'Provider Type',
       'Provider Subtype', 'Provider Status',
       'Individual Register Combinations',
       'Provider Early Years Register Flag',
       'Provider Compulsory Childcare Register Flag',
       'Provider Voluntary Childcare Register Flag', 'Registered Person URN',
       'Registered Person Name', 'Provider Name', 'Provider Address Line 1',
       'Provider Address Line 2', 'Provider Address Line 3', 'Provider Town',
       'Provider Postcode', 'Parliamentary Constituency', 'Local Authority',
       'Region', 'Ofsted Region', 'Deprivation Band', 'Places',
       'Places including Estimates', 'Most Recent Full: Inspection Number',
       'Most Recent Full: Inspection Date',
       'Most Recent Full: Overall Effectiveness', 'Quality of Education',
       'Behaviour and Attitudes', 'Personal Development',
       'Effectiveness of Leadership and Management',
       'Safeguarding is Effective?', 'CCR Requirements Suitability',
   

In [4]:
#confirm data types
df.dtypes

Provider URN                                    object
Registration Date                               object
Provider Type                                   object
Provider Subtype                                object
Provider Status                                 object
Individual Register Combinations                object
Provider Early Years Register Flag              object
Provider Compulsory Childcare Register Flag     object
Provider Voluntary Childcare Register Flag      object
Registered Person URN                           object
Registered Person Name                          object
Provider Name                                   object
Provider Address Line 1                         object
Provider Address Line 2                         object
Provider Address Line 3                         object
Provider Town                                   object
Provider Postcode                               object
Parliamentary Constituency                      object
Local Auth

In [5]:
#get info
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 62191 entries, 0 to 62190
Data columns (total 49 columns):
 #   Column                                       Non-Null Count  Dtype  
---  ------                                       --------------  -----  
 0   Provider URN                                 62191 non-null  object 
 1   Registration Date                            62191 non-null  object 
 2   Provider Type                                62191 non-null  object 
 3   Provider Subtype                             29389 non-null  object 
 4   Provider Status                              62191 non-null  object 
 5   Individual Register Combinations             62191 non-null  object 
 6   Provider Early Years Register Flag           62191 non-null  object 
 7   Provider Compulsory Childcare Register Flag  62191 non-null  object 
 8   Provider Voluntary Childcare Register Flag   62191 non-null  object 
 9   Registered Person URN                        27134 non-null  object 
 10

In [6]:
#select variables
df = df[[
    'Local Authority',
    'Most Recent Full: Overall Effectiveness',
    'Places'
]]

In [7]:
#filter by London
london_boroughs = [
    'Camden', 'Greenwich', 'Hackney', 'Hammersmith and Fulham',
    'Islington', 'Kensington and Chelsea', 'Lambeth', 'Lewisham',
    'Southwark', 'Tower Hamlets', 'Wandsworth', 'Westminster',
    'Barking and Dagenham', 'Barnet', 'Bexley', 'Brent',
    'Bromley', 'Croydon', 'Ealing', 'Enfield', 'Haringey',
    'Harrow', 'Havering', 'Hillingdon', 'Hounslow',
    'Kingston upon Thames', 'Merton', 'Newham', 'Redbridge',
    'Richmond upon Thames', 'Sutton', 'Waltham Forest',
    'City of London'
]
df = df[df['Local Authority'].isin(london_boroughs)]

In [8]:
#clean datatypes 
#first, convert rating to numeric (already encoded as 1–4)
df['score'] = pd.to_numeric(
    df['Most Recent Full: Overall Effectiveness'],
    errors='coerce'
)

In [9]:
#clean data types
#second, ensure Places is numeric
df['Places'] = pd.to_numeric(df['Places'], errors='coerce')

In [19]:
#check missing values
df[['score', 'Places']].isna().sum()

score     0
Places    0
dtype: int64

In [12]:
#create weighted values
df['weighted'] = df['score'] * df['Places']

In [13]:
#aggregate to borough level
df_la = df.groupby('Local Authority').agg({
    'weighted': 'sum',
    'Places': 'sum'
}).reset_index()

df_la['ofsted'] = df_la['weighted'] / df_la['Places']

In [14]:
#format output
df_clean = df_la[['Local Authority', 'ofsted']]

df_clean.columns = ['borough', 'ofsted']

df_clean = df_clean.sort_values(by='borough').reset_index(drop=True)

In [17]:
#validation
print(len(df_clean))
print(df_clean.head())

33
                borough    ofsted
0  Barking and Dagenham  2.013984
1                Barnet  1.890887
2                Bexley  1.951283
3                 Brent  2.030341
4               Bromley  1.821464


In [18]:
#Save clean dataset
df_clean.to_csv('../data_clean/ofsted_clean.csv', index=False)

In [20]:
#create total places per borough
df_places = df.groupby('Local Authority')['Places'].sum().reset_index()

df_places.columns = ['borough', 'total_places']

In [21]:
#validate
print("Rows:", len(df_places))
print(df_places.head())

Rows: 33
                borough  total_places
0  Barking and Dagenham          3361
1                Barnet          7451
2                Bexley          5029
3                 Brent          4779
4               Bromley          8144


In [22]:
#save new dataset for total places
df_places.to_csv('../data_clean/ofsted_places.csv', index=False)